In [ ]:
import torch
import torch.nn as nn
import importlib
import tiktoken
from torch.utils.data import Dataset, IterableDataset,  DataLoader
import os
import urllib.request
import json
import ssl
from functools import partial
import re
import copy
import time

# Install necessary library
import datasets
from datasets import load_dataset

#  "Tiny" config is ~28M parameters.
TINY_TM_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 384,
    "n_heads": 6,
    "n_layers": 6,
    "drop_rate": 0.1,
    "qkv_bias": False
}

TRAINING_CONFIG = {
  "pretraining_epoch": 6,
  "pretraining_maxsteps_epoch": 50000,
}

# --- Model Architecture Components ---

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -float("inf"))
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, -1)
        return self.out_proj(context_vec)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"], cfg["drop_rate"], cfg["n_heads"])
        self.ff = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])

    def forward(self, x):
        x = x + self.att(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class ThinkingMachine(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # lookup table to intiatize random embedding vector of size "emb_dim" for all tokens
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
         # lookup table to intiatize random pos embedding vector of size "emb_dim" for all tokens
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        self.apply(self._init_weights)

    def forward(self, in_idx):
        seq_len = in_idx.shape[1]
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.trf_blocks(x)
        return self.out_head(self.final_norm(x))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # Xavier/Glorot ensures the signal doesn't die out in deep layers
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, (nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


# --- Training Logic ---

def text_to_token_ids(text, tokenizer, device):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0).to(device) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


# 1. Mount Drive
#drive.mount('/content/drive')
#from google.colab import drive
# 2. Setup Drive storage path
#SAVE_CHECKPOINT_DIR = "/content/drive/MyDrive/ThinkingMachine_Checkpoints"
#SAVE_MODEL_DIR = "/content/drive/MyDrive/ThinkingMachine_Models"


# Setup LOCAL checkpoint directory (instead of Google Drive)
SAVE_CHECKPOINT_DIR = "./model_checkpoints"
SAVE_MODEL_DIR = "./models"

os.makedirs(SAVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAVE_MODEL_DIR, exist_ok=True)  


def load_checkpoint_from_drive(model, optimizer):
    """Checks if a checkpoint exists and loads it."""
    checkpoint_path = os.path.join(SAVE_CHECKPOINT_DIR, "model_latest.pth")
    if os.path.exists(checkpoint_path):
        print(f"🔄 Found checkpoint at {checkpoint_path}. Resuming...")
        checkpoint = torch.load(checkpoint_path)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        start_step = checkpoint['global_step']
        epoch = checkpoint['epoch']

        return start_step, epoch
    else:
        print("🌱 No checkpoint found. Starting training from scratch.")
        return 0,0

def save_checkpoint_to_drive(model, optimizer, global_step, epoch):
    # We save as 'latest' for easy resuming
    path = os.path.join(SAVE_CHECKPOINT_DIR, "model_latest.pth")

    torch.save({
        'global_step': global_step,
        'epoch':epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, path)
    print(f"💾 Checkpoint saved to Drive for epoch {epoch} at step {global_step}")

def load_and_resize_model(model, optimizer, model_name):
    path = os.path.join(SAVE_MODEL_DIR, model_name)
    if not os.path.exists(path):
        print("🌱 No model found. Starting from scratch.")
        return

    print(f"Performing Surgery on {path}...")
    checkpoint = torch.load(path, map_location=torch.device('cpu'))

    # Extract the state dictionary
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint

    # --- SURGERY STEP 1: Fix Positional Embeddings ---
    # Get the weight from the file
    old_pos = state_dict['pos_emb.weight']
    new_pos = model.pos_emb.weight.data

    if old_pos.shape != new_pos.shape:
        print(f"⚠️ Resizing Pos Embeddings: {old_pos.shape} -> {new_pos.shape}")

        # Create a new tensor with the TARGET shape (randomly initialized)
        resized_pos = torch.randn_like(new_pos) * 0.02

        # Copy the OLD learned weights into the first part of the matrix
        # We keep the knowledge for positions 0-127!
        n_old = old_pos.shape[0]
        resized_pos[:n_old, :] = old_pos

        # Update the state_dict
        state_dict['pos_emb.weight'] = resized_pos

        # --- SURGERY STEP 2: Fix Attention Masks ---
        # Your code uses a buffer called 'mask'. The old mask is 128x128.
        # The new model has a 256x256 mask.
        # Since 'mask' is a fixed buffer (not learned), we can just DELETE it from the load.
        # The new model will keep its own correct 256x256 mask.
        keys_to_delete = [k for k in state_dict.keys() if 'mask' in k]
        for k in keys_to_delete:
            del state_dict[k]

    # Load the fixed state_dict
    # strict=False allows us to ignore the missing 'mask' keys we just deleted
    model.load_state_dict(state_dict, strict=False)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print("✅ Surgery Complete")
    return checkpoint['global_step'], checkpoint['epoch']


def save_model(model, optimizer, model_name):
    path = os.path.join(SAVE_MODEL_DIR, model_name)
    temp_path = path + ".tmp" # Save to a temporary file first

    print(f"💾 Saving to {path}...")

    # Save a dictionary (it's safer and more flexible)
    state = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }

    # Step 1: Save to a temporary file
    torch.save(state, temp_path)

    # Step 2: Force a sync to physical storage
    os.sync()

    # Step 3: Rename temp file to final filename (Atomic move)
    if os.path.exists(temp_path):
        os.replace(temp_path, path)
        print("✅ Save verified and complete.")

def load_model(model, model_name):
    path = os.path.join(SAVE_MODEL_DIR, model_name)
    if os.path.exists(path):
        print(f"🔄 Loading weights from {path}...")
        # Add map_location to handle CPU/GPU differences
        checkpoint = torch.load(path, map_location=torch.device('cpu'))
        # If you saved using 'save_checkpoint_to_drive',
        # the weights are hidden inside the 'model_state_dict' key
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            # If you saved using just torch.save(model.state_dict()...)
            model.load_state_dict(checkpoint)
        print("✅ Weights loaded successfully.")
    else:
        print("🌱 No pretrained model found.")

class StreamingTextDataset(IterableDataset):
    def __init__(self, dataset_source, tokenizer, max_length):
        self.dataset_source = dataset_source
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __iter__(self):
        for item in self.dataset_source:
            ids = self.tokenizer.encode(item['text'], allowed_special={'<|endoftext|>'})
            for i in range(0, len(ids) - self.max_length, self.max_length):
                yield torch.tensor(ids[i:i+self.max_length]), torch.tensor(ids[i+1:i+self.max_length+1])

# --- Sample Generation with Temperature Sampling ---
def generate_creative(model, tokenizer, prompt, max_tokens, context_size, device, temp=0.8, top_k=10):
    model.eval()
    idx = text_to_token_ids(prompt, tokenizer,device)
     # ID for <|endoftext|>
    eos_id = tokenizer.encode('<|endoftext|>', allowed_special={'<|endoftext|>'})[0]

    for _ in range(max_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
          logits = model(idx_cond)[:, -1, :]

          # --- Apply Temperature ---
          logits = logits / temp

          # --- Top-K Filtering ---
          if top_k is not None:
              # Keep only the top k values, set others to -infinity
              top_logits, _ = torch.topk(logits, top_k)
              min_val = top_logits[:, [-1]]
              logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(device), logits)

          # --- Sample (not argmax) ---
          probas = torch.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probas, num_samples=1)

          # Check for End of Text token
          if idx_next.item() == eos_id:
              print("\n[Generated EOS token - Stopping]")
              break

          # --- Append the next token ---
          # Corrected the cat syntax: ( (tensors), dim )
          idx = torch.cat((idx, idx_next), dim=1)

    return token_ids_to_text(idx, tokenizer)


def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Training on: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

    # Setup
    torch.manual_seed(123)
    batch_size = 16
    max_steps_per_epoch = TRAINING_CONFIG["pretraining_maxsteps_epoch"]
    tokenizer = tiktoken.get_encoding("gpt2")
    model = ThinkingMachine(TINY_TM_CONFIG).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004)

    # Resume Logic
    start_step, start_epoch = load_checkpoint_from_drive(model, optimizer)
    end_epoch = TRAINING_CONFIG["pretraining_epoch"]
    dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="train")
    val_data = dataset.take(1000)
    val_loader = DataLoader(StreamingTextDataset(val_data, tokenizer, 128), batch_size=batch_size)
    eval_prompt = "write me a good story in maximum 50 words"
    start_time = time.time()
    try:
        for epoch in range(start_epoch, end_epoch):
            # Calculate sequences to skip: (Previous Epochs * Max Steps) + Current Steps
            sequences_to_skip = (epoch * max_steps_per_epoch) + start_step
            train_data = dataset.skip(1000 + sequences_to_skip)


            train_loader = DataLoader(
                StreamingTextDataset(train_data, tokenizer, 128),
                batch_size=batch_size,
                pin_memory=True
            )

            epoch_step = start_step
            for batch_idx, (input_batch, target_batch) in enumerate(train_loader):
                if epoch_step >= max_steps_per_epoch: break

                model.train()
                optimizer.zero_grad()
                loss = calc_loss_batch(input_batch, target_batch, model, device)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                # Stats & Eval
                if epoch_step % 1000 == 0 and epoch_step > 0:
                    with torch.no_grad():
                        model.eval()
                        v_batch = next(iter(val_loader))
                        v_loss = calc_loss_batch(v_batch[0], v_batch[1], model, device)
                        print(f"[{time.strftime('%H:%M:%S')}] Ep {epoch} Step {epoch_step} | Loss: {loss.item():.4f} | Val: {v_loss.item():.4f}")

                        if epoch_step % 10000 == 0:
                            save_checkpoint_to_drive(model, optimizer, epoch_step, epoch)
                            print(f"\n✅ Eval TEST: {generate_creative(model, tokenizer, eval_prompt, 100, TINY_TM_CONFIG["context_length"], device, 1.2, 3)}\n")

                epoch_step += 1

            start_step = 0 # Reset for next epoch
            print(f"--- Finished Epoch {epoch} ---")
            save_checkpoint_to_drive(model, optimizer, 0, epoch + 1)

    except KeyboardInterrupt:
        print("\nInterrupted")

    save_model(model, optimizer, "pretrained_model.pth")
    print(f"Done! Total time: {(time.time() - start_time) / 3600:.2f} hours")


if __name__ == "__main__":
    run_training()
